In [17]:
from binance.client import Client
import pandas as pd
import time
from datetime import datetime, timedelta
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import Rectangle

# 设置matplotlib
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

def get_usdt_binance_symbols(client):
    """Get all available USDT trading pairs on Binance"""
    exchange_info = client.get_exchange_info()
    symbols = []
    
    for symbol_info in exchange_info['symbols']:
        if symbol_info['status'] == 'TRADING' and symbol_info['symbol'].endswith('USDT'):
            symbols.append(symbol_info['symbol'])
    
    print(f"Found {len(symbols)} USDT trading pairs on Binance")
    return symbols

def get_historical_volume(client, symbol, lookback_days=17):
    """Get historical daily volume data for a symbol"""
    try:
        start_time = (datetime.now() - timedelta(days=lookback_days+1)).strftime("%d %b %Y %H:%M:%S")
        
        klines = client.get_historical_klines(
            symbol=symbol,
            interval=Client.KLINE_INTERVAL_1DAY,
            start_str=start_time
        )
        
        df = pd.DataFrame(klines, columns=[
            'open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time',
            'quote_asset_volume', 'number_of_trades', 'taker_buy_base', 'taker_buy_quote', 'ignored'
        ])
        
        df['date'] = pd.to_datetime(df['open_time'], unit='ms')
        df['volume'] = df['volume'].astype(float)
        
        return df
        
    except Exception as e:
        print(f"Error getting data for {symbol}: {e}")
        return None

def check_volume_spike(df, days_to_check=10, previous_days=7, threshold=5.0):
    """Check if volume in any of the last 'days_to_check' days is greater than
    'threshold' times the average volume of the previous 'previous_days' days"""
    if df is None or len(df) < (days_to_check + previous_days):
        return False, {}
    
    df = df.sort_values('date')
    recent_data = df.tail(days_to_check + previous_days)
    result = {"has_spike": False, "spike_info": {}}
    
    for i in range(previous_days, previous_days + days_to_check):
        if i >= len(recent_data):
            break
            
        current_day = recent_data.iloc[i]
        prev_days = recent_data.iloc[i-previous_days:i]
        avg_volume = prev_days['volume'].mean()
        
        if avg_volume > 0 and current_day['volume'] > threshold * avg_volume:
            current_date = current_day['date'].strftime('%Y-%m-%d')
            
            result["has_spike"] = True
            result["spike_info"] = {
                "date": current_date,
                "volume": current_day['volume'],
                "avg_prev_volume": avg_volume,
                "ratio": current_day['volume'] / avg_volume if avg_volume > 0 else float('inf'),
            }
            break
    
    return result["has_spike"], result["spike_info"]

def find_volume_spike_coins(api_key="", api_secret="", threshold=5.0, max_symbols=None):
    """Find USDT trading pairs on Binance with volume spikes"""
    client = Client(api_key, api_secret)
    all_symbols = get_usdt_binance_symbols(client)
    
    if max_symbols is not None and max_symbols > 0:
        all_symbols = all_symbols[:max_symbols]
        print(f"Checking {len(all_symbols)} USDT symbols...")
    
    spike_coins = []
    
    for i, symbol in enumerate(all_symbols):
        try:
            if i % 10 == 0:
                print(f"Processing {i}/{len(all_symbols)}: {symbol}")
            
            df = get_historical_volume(client, symbol)
            has_spike, spike_info = check_volume_spike(df, threshold=threshold)
            
            if has_spike:
                spike_coins.append({
                    "symbol": symbol,
                    "spike_date": spike_info["date"],
                    "volume": spike_info["volume"],
                    "avg_prev_volume": spike_info["avg_prev_volume"],
                    "ratio": spike_info["ratio"]
                })
                print(f"✅ Found spike for {symbol}: {spike_info['ratio']:.2f}x on {spike_info['date']}")
            
            time.sleep(0.1)
            
        except Exception as e:
            print(f"Error processing {symbol}: {e}")
    
    spike_coins.sort(key=lambda x: x["ratio"], reverse=True)
    return spike_coins, client

def print_results(spike_coins):
    """Print results directly"""
    if not spike_coins:
        print("No volume spike coins found.")
        return
    
    print("\n" + "="*80)
    print(f"FOUND {len(spike_coins)} USDT PAIRS WITH VOLUME SPIKES > 5X THE PREVIOUS 7-DAY AVERAGE")
    print("="*80)
    print(f"{'RANK':<6}{'SYMBOL':<12}{'SPIKE DATE':<12}{'VOLUME':<20}{'AVG VOLUME':<20}{'RATIO':<10}")
    print("-"*80)
    
    for i, coin in enumerate(spike_coins, 1):
        print(f"{i:<6}{coin['symbol']:<12}{coin['spike_date']:<12}{coin['volume']:<20.2f}{coin['avg_prev_volume']:<20.2f}{coin['ratio']:<10.2f}x")
    
    print("\n" + "="*80)
    print("SUMMARY: TOP 10 USDT PAIRS WITH HIGHEST VOLUME SPIKES")
    print("="*80)
    
    for i, coin in enumerate(spike_coins[:10], 1):
        print(f"{i}. {coin['symbol']}: {coin['ratio']:.2f}x volume spike on {coin['spike_date']}")

def plot_candlestick(ax, df, width=0.7):
    """
    绘制标准的日本蜡烛图
    """
    for idx, row in df.iterrows():
        # 确定颜色
        if row['close'] >= row['open']:
            color = '#26a69a'  # 绿色（上涨）
            facecolor = '#26a69a'
        else:
            color = '#ef5350'  # 红色（下跌）
            facecolor = '#ef5350'
        
        # 计算x坐标（使用数值日期）
        x = mdates.date2num(row['date'])
        
        # 绘制影线（最高价到最低价）
        ax.plot([x, x], [row['low'], row['high']], color=color, linewidth=0.8)
        
        # 绘制实体（开盘价到收盘价）
        body_bottom = min(row['open'], row['close'])
        body_height = abs(row['close'] - row['open'])
        
        if body_height > 0:
            rect = Rectangle((x - width/2, body_bottom), width, body_height,
                           facecolor=facecolor, edgecolor=color, linewidth=0.8)
            ax.add_patch(rect)
        else:
            # 如果开盘价等于收盘价，画一条线
            ax.plot([x - width/2, x + width/2], [row['close'], row['close']], color=color, linewidth=1)

def plot_all_to_pdf(spike_coins, client, top_n=20, days_to_show=30, output_filename='volume_spike_analysis.pdf'):
    """
    Plot all top N coins to a single PDF file with standard candlestick charts
    """
    if not spike_coins:
        print("No volume spike coins found.")
        return
    
    top_coins = spike_coins[:top_n]
    print(f"\nPlotting top {len(top_coins)} coins to PDF...")
    print(f"Output file: {output_filename}")
    print(f"Showing last {days_to_show} days for each coin")
    
    # Create PDF file
    with PdfPages(output_filename) as pdf:
        # Page 1: Summary table
        fig, ax = plt.subplots(figsize=(14, 10))
        ax.axis('tight')
        ax.axis('off')
        
        # Prepare table data
        table_data = []
        for i, coin in enumerate(top_coins, 1):
            table_data.append([
                i,
                coin['symbol'],
                coin['spike_date'],
                f"{coin['ratio']:.2f}x",
                f"{coin['volume']:,.0f}",
                f"{coin['avg_prev_volume']:,.0f}"
            ])
        
        columns = ['Rank', 'Symbol', 'Spike Date', 'Volume Ratio', 'Spike Volume', 'Avg Volume (7d)']
        table = ax.table(cellText=table_data, colLabels=columns, cellLoc='center', loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1.2, 1.5)
        
        title = f'Top {len(top_coins)} Coins with Volume Spikes (>5x 7-day Average)\n'
        title += f'Generated on {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}'
        ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
        
        pdf.savefig(fig, bbox_inches='tight')
        plt.close()
        
        # Plot each coin
        for idx, coin_info in enumerate(top_coins, 1):
            symbol = coin_info['symbol']
            spike_date = coin_info['spike_date']
            spike_ratio = coin_info['ratio']
            
            print(f"Plotting {idx}/{len(top_coins)}: {symbol}")
            
            try:
                # Get historical data
                end_date = datetime.now()
                start_date = end_date - timedelta(days=days_to_show + 5)
                
                klines = client.get_historical_klines(
                    symbol=symbol,
                    interval=Client.KLINE_INTERVAL_1DAY,
                    start_str=start_date.strftime("%d %b %Y %H:%M:%S"),
                    end_str=end_date.strftime("%d %b %Y %H:%M:%S")
                )
                
                if not klines:
                    print(f"  ⚠️ No data for {symbol}, skipping")
                    continue
                
                # Convert to DataFrame
                df = pd.DataFrame(klines, columns=[
                    'open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time',
                    'quote_asset_volume', 'number_of_trades', 'taker_buy_base', 'taker_buy_quote', 'ignored'
                ])
                
                # Data type conversion
                df['date'] = pd.to_datetime(df['open_time'], unit='ms')
                df['open'] = df['open'].astype(float)
                df['high'] = df['high'].astype(float)
                df['low'] = df['low'].astype(float)
                df['close'] = df['close'].astype(float)
                df['volume'] = df['volume'].astype(float)
                
                # Keep only last days_to_show days
                df = df.tail(days_to_show)
                
                if len(df) < 10:
                    print(f"  ⚠️ Insufficient data for {symbol} ({len(df)} days), skipping")
                    continue
                
                # Create figure with specific size
                fig = plt.figure(figsize=(14, 9))
                
                # Create subplots with proper spacing
                # [left, bottom, width, height]
                ax1 = plt.axes([0.08, 0.30, 0.88, 0.60])  # Price chart
                ax2 = plt.axes([0.08, 0.10, 0.88, 0.15])  # Volume chart
                
                # Plot candlesticks
                plot_candlestick(ax1, df, width=0.7)
                
                # Mark spike day
                spike_date_dt = datetime.strptime(spike_date, '%Y-%m-%d')
                spike_found = False
                if spike_date_dt in df['date'].values:
                    spike_found = True
                    spike_idx = df[df['date'] == spike_date_dt].index[0]
                    spike_price = df.loc[spike_idx, 'high']
                    # Add vertical line
                    ax1.axvline(x=spike_date_dt, color='orange', linestyle='--', linewidth=1.5, alpha=0.7)
                    # Add text annotation
                    ax1.annotate(f'{spike_ratio:.1f}x', 
                               xy=(spike_date_dt, spike_price),
                               xytext=(5, 5), 
                               textcoords='offset points',
                               fontsize=10, 
                               color='orange',
                               weight='bold',
                               alpha=0.9,
                               bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
                
                # Configure price chart
                title_text = f'Rank #{idx}: {symbol} - Volume Spike: {spike_ratio:.2f}x on {spike_date}\n'
                title_text += f'Period: {df["date"].iloc[0].strftime("%Y-%m-%d")} to {df["date"].iloc[-1].strftime("%Y-%m-%d")}'
                ax1.set_title(title_text, fontsize=13, fontweight='bold', pad=15)
                ax1.set_ylabel('Price (USDT)', fontsize=11)
                ax1.grid(True, alpha=0.3, linestyle='--')
                ax1.set_xlim(df['date'].iloc[0] - pd.Timedelta(days=0.5), 
                           df['date'].iloc[-1] + pd.Timedelta(days=0.5))
                
                # Format x-axis for price chart (hide labels, will show on volume chart)
                ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
                ax1.tick_params(axis='x', labelbottom=False)
                
                # Plot volume
                for i, row in df.iterrows():
                    if row['close'] >= row['open']:
                        color = '#26a69a'  # 绿色
                    else:
                        color = '#ef5350'  # 红色
                    
                    ax2.bar(row['date'], row['volume'], width=0.7, color=color, alpha=0.7, align='center')
                
                # Mark spike day volume
                if spike_found:
                    spike_volume = df.loc[spike_idx, 'volume']
                    ax2.axvline(x=spike_date_dt, color='orange', linestyle='--', linewidth=1.5, alpha=0.7)
                    ax2.annotate(f'{spike_ratio:.1f}x', 
                               xy=(spike_date_dt, spike_volume),
                               xytext=(5, 5), 
                               textcoords='offset points',
                               fontsize=10, 
                               color='orange',
                               weight='bold',
                               alpha=0.9,
                               bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
                
                # Configure volume chart
                ax2.set_xlabel('Date', fontsize=11)
                ax2.set_ylabel('Volume', fontsize=11)
                ax2.grid(True, alpha=0.3, linestyle='--')
                ax2.set_xlim(df['date'].iloc[0] - pd.Timedelta(days=0.5), 
                           df['date'].iloc[-1] + pd.Timedelta(days=0.5))
                
                # Format x-axis dates
                ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
                
                # Rotate date labels for better readability
                plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')
                
                # No need for tight_layout when using manual positioning
                
                # Save to PDF
                pdf.savefig(fig, bbox_inches='tight')
                plt.close(fig)
                
                print(f"  ✅ Added to PDF")
                
                # Small delay to avoid rate limits
                time.sleep(0.2)
                
            except Exception as e:
                print(f"  ❌ Error plotting {symbol}: {e}")
                import traceback
                traceback.print_exc()
                plt.close()
                continue
        
        # Last page: Summary and disclaimer
        fig, ax = plt.subplots(figsize=(14, 10))
        ax.axis('tight')
        ax.axis('off')
        
        avg_ratio = sum(c['ratio'] for c in top_coins) / len(top_coins)
        
        summary_text = f"""
Volume Spike Analysis Report

Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
Condition: Volume > 5x previous 7-day average
Analysis: Top {len(top_coins)} trading pairs
Chart data: Last {days_to_show} days for each pair

Summary Statistics:
- Total pairs analyzed: {len(top_coins)}
- Highest volume spike: {top_coins[0]['ratio']:.2f}x ({top_coins[0]['symbol']} on {top_coins[0]['spike_date']})
- Average volume spike: {avg_ratio:.2f}x

Chart Legend:
- Green candles: Close price > Open price (bullish)
- Red candles: Close price < Open price (bearish)
- Orange dashed line: Volume spike day
- Orange text: Spike multiplier

Methodology:
- Spike detection: Volume on spike day > 5x average volume of previous 7 days
- Data source: Binance spot market daily candles
- Chart period: Last {days_to_show} days up to report generation

Disclaimer:
This report is for informational purposes only and does not constitute investment advice.
Past performance does not guarantee future results.
"""
        
        ax.text(0.1, 0.5, summary_text, transform=ax.transAxes, fontsize=11, verticalalignment='center',
               fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        pdf.savefig(fig, bbox_inches='tight')
        plt.close()
    
    print(f"\n✅ PDF saved successfully: {output_filename}")
    print(f"   Total pages: {len(top_coins) + 2} (Summary + {len(top_coins)} charts + Disclaimer)")

if __name__ == "__main__":
    API_KEY = ""
    API_SECRET = ""
    
    print("Starting volume spike analysis...")
    print("="*60)
    
    # Find volume spike coins
    spike_coins, client = find_volume_spike_coins(
        api_key=API_KEY,
        api_secret=API_SECRET,
        threshold=5.0,
        max_symbols=None  # Check first 200 symbols for faster testing
    )
    
    # Print results
    print_results(spike_coins)
    
    # Plot to PDF
    if spike_coins:
        plot_all_to_pdf(
            spike_coins, 
            client, 
            top_n=20,           # Top 20 coins
            days_to_show=30,    # Show 30 days of data
            output_filename='volume_spike_analysis.pdf'
        )
        
        print("\n✅ Analysis complete!")
    else:
        print("\nNo volume spike coins found.")

Starting volume spike analysis...
Found 432 USDT trading pairs on Binance
Processing 0/432: BTCUSDT
Processing 10/432: XLMUSDT
✅ Found spike for ONTUSDT: 8.00x on 2026-05-06
✅ Found spike for ONGUSDT: 6.11x on 2026-05-05
Processing 20/432: ZILUSDT
✅ Found spike for ZILUSDT: 8.28x on 2026-05-06
✅ Found spike for BATUSDT: 12.03x on 2026-05-03
✅ Found spike for DASHUSDT: 8.42x on 2026-05-04
Processing 30/432: ATOMUSDT
✅ Found spike for ANKRUSDT: 18.02x on 2026-05-06
✅ Found spike for MTLUSDT: 6.86x on 2026-05-05
Processing 40/432: CVCUSDT
✅ Found spike for STXUSDT: 33.27x on 2026-05-05
✅ Found spike for KAVAUSDT: 8.48x on 2026-05-02
Processing 50/432: RLCUSDT
✅ Found spike for RLCUSDT: 14.66x on 2026-05-03
✅ Found spike for FTTUSDT: 15.39x on 2026-05-05
Processing 60/432: CTSIUSDT
✅ Found spike for HIVEUSDT: 139.10x on 2026-05-05
✅ Found spike for KNCUSDT: 61.53x on 2026-05-02
✅ Found spike for SCUSDT: 10.23x on 2026-05-06
Processing 70/432: DGBUSDT
✅ Found spike for STORJUSDT: 13.17x on 